# HindiMix-Noisy — ByT5 Proposed Model (All Noise Levels)

**Runs:** clean, low, medium, high

**Setup before running:**
1. Kaggle → Settings → Accelerator → **GPU T4 x2** (or P100)
2. Upload `data/final/` CSVs as a Kaggle Dataset named `hindimix-data`
   - Go to kaggle.com → Datasets → New Dataset → upload all 6 CSVs
   - Then add it to this notebook: Add Data → Your Datasets → hindimix-data
3. Run all cells in order

**Results saved to:** `/kaggle/working/` → download from Output tab when done

## Step 1: Install dependencies

In [ ]:
!pip install -q transformers==4.40.0 sentencepiece jellyfish accelerate
import torch
print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB' if torch.cuda.is_available() else '')

## Step 2: Check data paths

In [ ]:
import os

# Kaggle datasets are mounted at /kaggle/input/<dataset-name>/
# Change this to match your dataset name on Kaggle
DATASET_NAME = 'hindimix-data'   # <-- change if your dataset has a different name

DATA_DIR = f'/kaggle/input/{DATASET_NAME}'
if not os.path.exists(DATA_DIR):
    # fallback: check if files are directly in /kaggle/input/
    for d in os.listdir('/kaggle/input/'):
        if 'hindimix' in d.lower() or 'train' in d.lower():
            DATA_DIR = f'/kaggle/input/{d}'
            break

RESULTS_DIR = '/kaggle/working/results'
CKPT_DIR    = '/kaggle/working/checkpoints'
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

print('Data dir:', DATA_DIR)
print('Files found:')
for f in sorted(os.listdir(DATA_DIR)):
    rows = sum(1 for _ in open(f'{DATA_DIR}/{f}')) - 1
    print(f'  {f}: {rows:,} rows')

## Step 3: Model architecture (ByT5 + Phonetic + Noise-Aware Attention)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import T5EncoderModel
import jellyfish


class PhoneticFeatureExtractor(nn.Module):
    def __init__(self, feature_dim=64):
        super().__init__()
        self.proj = nn.Linear(4, feature_dim)

    def compute_phonetic_features(self, tokens):
        features = []
        for token in tokens:
            soundex   = jellyfish.soundex(token)   if token.isalpha() else '0000'
            metaphone = jellyfish.metaphone(token) if token.isalpha() else ''
            feat = [
                hash(soundex) % 1000 / 1000.0,
                len(metaphone) / 10.0,
                len(token) / 20.0,
                len(set(token)) / max(len(token), 1),
            ]
            features.append(feat)
        return torch.tensor(features, dtype=torch.float32)

    def forward(self, tokens):
        feats = self.compute_phonetic_features(tokens)
        feats = feats.to(self.proj.weight.device)   # fix: move to GPU
        return self.proj(feats)


class NoiseAwareAttention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.gate = nn.Linear(hidden_size, 1)

    def forward(self, hidden_states):
        gates = torch.sigmoid(self.gate(hidden_states))
        return hidden_states * gates


class NoiseRobustHateDetector(nn.Module):
    def __init__(self, num_labels=2, phonetic_dim=64, dropout=0.1):
        super().__init__()
        self.encoder      = T5EncoderModel.from_pretrained('google/byt5-small')
        self.hidden_size  = self.encoder.config.d_model
        self.phonetic_extractor = PhoneticFeatureExtractor(feature_dim=phonetic_dim)
        self.phonetic_proj      = nn.Linear(phonetic_dim, self.hidden_size)
        self.noise_attention    = NoiseAwareAttention(self.hidden_size)
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(self.hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_labels),
        )

    def forward(self, input_ids, attention_mask, token_strings=None, labels=None):
        enc_out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        hidden  = enc_out.last_hidden_state

        if token_strings is not None:
            phon_feats = []
            for tokens in token_strings:
                feat    = self.phonetic_extractor(tokens)
                seq_len = hidden.size(1)
                if feat.size(0) < seq_len:
                    pad  = torch.zeros(seq_len - feat.size(0), feat.size(1), device=hidden.device)
                    feat = torch.cat([feat, pad], dim=0)
                else:
                    feat = feat[:seq_len]
                phon_feats.append(feat)
            phon_feats = torch.stack(phon_feats).to(hidden.device)
            hidden     = hidden + self.phonetic_proj(phon_feats)

        hidden = self.noise_attention(hidden)
        mask   = attention_mask.unsqueeze(-1).float()
        pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
        logits = self.classifier(self.dropout(pooled))

        loss = F.cross_entropy(logits, labels) if labels is not None else None
        return {'loss': loss, 'logits': logits}


print('Model class defined.')
print('ByT5-small hidden size will be 1472.')

## Step 4: Training utilities

In [ ]:
import json
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from torch.optim import AdamW
from torch.amp import autocast, GradScaler
from sklearn.metrics import f1_score, classification_report
from tqdm.notebook import tqdm

DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_NAME = 'google/byt5-small'
EPOCHS     = 5
BATCH_SIZE = 4
MAX_LEN    = 128
LR         = 3e-5
scaler     = GradScaler('cuda')


class HateSpeechDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.encodings = tokenizer(
            texts, max_length=MAX_LEN, padding='max_length',
            truncation=True, return_tensors='pt'
        )
        self.labels = torch.tensor(labels, dtype=torch.long)
        self.texts  = texts

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids':      self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels':         self.labels[idx],
            'text':           self.texts[idx],
        }


def collate_fn(batch):
    return {
        'input_ids':      torch.stack([b['input_ids']      for b in batch]),
        'attention_mask': torch.stack([b['attention_mask'] for b in batch]),
        'labels':         torch.stack([b['labels']         for b in batch]),
        'texts':          [b['text'] for b in batch],
    }


def load_data(noise_level):
    train_df = pd.read_csv(f'{DATA_DIR}/train.csv').dropna(subset=['text','label'])
    val_df   = pd.read_csv(f'{DATA_DIR}/val.csv').dropna(subset=['text','label'])
    if noise_level != 'all':
        train_df = train_df[train_df['noise_level'].isin(['clean', noise_level])]
    test_path = f'{DATA_DIR}/test_clean.csv' if noise_level == 'clean' \
                else f'{DATA_DIR}/test_noisy_{noise_level}.csv'
    test_df = pd.read_csv(test_path).dropna(subset=['text','label'])
    print(f'  Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')
    return train_df, val_df, test_df


def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0
    for batch in tqdm(loader, desc='  train', leave=False):
        optimizer.zero_grad()
        token_strings = [t.split() for t in batch['texts']]
        with autocast('cuda'):
            out = model(
                input_ids=batch['input_ids'].to(DEVICE),
                attention_mask=batch['attention_mask'].to(DEVICE),
                token_strings=token_strings,
                labels=batch['labels'].to(DEVICE),
            )
        scaler.scale(out['loss']).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total_loss += out['loss'].item()
    return total_loss / len(loader)


def evaluate(model, loader):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc='  eval', leave=False):
            token_strings = [t.split() for t in batch['texts']]
            with autocast('cuda'):
                out = model(
                    input_ids=batch['input_ids'].to(DEVICE),
                    attention_mask=batch['attention_mask'].to(DEVICE),
                    token_strings=token_strings,
                )
            preds.extend(out['logits'].argmax(dim=-1).cpu().tolist())
            targets.extend(batch['labels'].tolist())
    return f1_score(targets, preds, average='macro'), preds, targets


print('Utilities ready. Device:', DEVICE)
print(f'Batch size: {BATCH_SIZE} | Max len: {MAX_LEN} | fp16: enabled')

## Step 5: Train ByT5 — all 4 noise levels
~90 min per level on T4 → ~6 hrs total. Each level saves a checkpoint + JSON result.

In [ ]:
tokenizer   = AutoTokenizer.from_pretrained(MODEL_NAME)
all_results = []

for noise in ['clean', 'low', 'medium', 'high']:
    print(f'\n{"="*60}')
    print(f'ByT5 | noise={noise}')
    print(f'{"="*60}')

    train_df, val_df, test_df = load_data(noise)

    train_loader = DataLoader(
        HateSpeechDataset(train_df['text'].tolist(), train_df['label'].tolist(), tokenizer),
        batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn
    )
    val_loader = DataLoader(
        HateSpeechDataset(val_df['text'].tolist(), val_df['label'].tolist(), tokenizer),
        batch_size=BATCH_SIZE, collate_fn=collate_fn
    )
    test_loader = DataLoader(
        HateSpeechDataset(test_df['text'].tolist(), test_df['label'].tolist(), tokenizer),
        batch_size=BATCH_SIZE, collate_fn=collate_fn
    )

    model = NoiseRobustHateDetector(num_labels=2).to(DEVICE)
    model.encoder.gradient_checkpointing_enable()   # saves ~30% VRAM

    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    total_steps = len(train_loader) * EPOCHS
    scheduler   = get_linear_schedule_with_warmup(optimizer, total_steps // 10, total_steps)

    ckpt_path   = f'{CKPT_DIR}/byt5_{noise}.pt'
    best_val_f1 = 0.0

    for epoch in range(EPOCHS):
        loss    = train_epoch(model, train_loader, optimizer, scheduler)
        val_f1, _, _ = evaluate(model, val_loader)
        print(f'  Epoch {epoch+1}/{EPOCHS} | loss: {loss:.4f} | val F1: {val_f1:.4f}')
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save(model.state_dict(), ckpt_path)
            print(f'    -> best saved')

    model.load_state_dict(torch.load(ckpt_path))
    test_f1, test_preds, test_targets = evaluate(model, test_loader)
    print(f'\n  Test F1 (macro): {test_f1:.4f}')
    print(classification_report(test_targets, test_preds, target_names=['Non-hate', 'Hate']))

    result = {
        'model': 'byt5_proposed',
        'noise_level': noise,
        'test_f1_macro': round(test_f1, 4),
        'best_val_f1':   round(best_val_f1, 4),
        'epochs': EPOCHS, 'lr': LR, 'batch_size': BATCH_SIZE
    }
    out = f'{RESULTS_DIR}/byt5_proposed_{noise}.json'
    with open(out, 'w') as f:
        json.dump(result, f, indent=2)
    print(f'  Saved -> {out}')
    all_results.append(result)

    del model
    torch.cuda.empty_cache()

print('\nAll noise levels done!')

## Step 6: Summary + download results

In [ ]:
print('\n--- ByT5 Results Summary ---')
print(f'{"noise":<10} {"val F1":>8} {"test F1":>9}')
for r in all_results:
    print(f"  {r['noise_level']:<8} {r['best_val_f1']:>8.4f} {r['test_f1_macro']:>9.4f}")

# Zip all result JSONs
import shutil
shutil.make_archive('/kaggle/working/byt5_results', 'zip', RESULTS_DIR)
print('\nResults zipped -> /kaggle/working/byt5_results.zip')
print('Download from the Output tab on the right ->')